# Load Libraries and Data

In [6]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms

from utils.neural_networks_utils import *
from utils.random_search_utils import random_search, get_best_hyper_parameters, run_random_search
from utils.data_loaders import load_dataset
from utils.utils import *

import os
import pandas as pd

from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler

import warnings
import numpy as np
import importlib

import copy

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True

In [7]:
# Check and create directories for saving processed data
cur_dir = os.getcwd()
model_save_loc = os.path.join(cur_dir, "models/DFNN")
check_create_directory(model_save_loc)
result_save_loc = os.path.join(cur_dir, "results/DFNN")
check_create_directory(result_save_loc)


Directory created: c:\Users\20225378\Desktop\ML Paper\Python Code\models/DFNN
Directory created: c:\Users\20225378\Desktop\ML Paper\Python Code\results/DFNN


In [3]:
# This script processes and fits DFNN models used in the study.

### Load Mean Data and Images

In [27]:
# Get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and process mean data from IDEAS (Deniz et al., 2026)
mean_data = pd.read_csv(os.path.join(os.path.dirname(cur_dir), "Visual Features/combined_visual_features.csv"))
# Get number of visual features in the dataset
number_of_visual_features =  len(mean_data[mean_data.columns.difference(["image", "valence", "Unnamed: 0", "room_type", "approach", "tense", "energetic"])].columns)

In [28]:
# Set the parameter space for the random search grid
param_grid = {
    'net__lr': [0.01,0.001,0.0001],
    'net__optimizer__weight_decay': [0.01,0.001,0.0001],
    'net__module__drop_out_rate': [0.1,0.2,0.3,0.4, 0.5],
    'net__batch_size':[32,64,128,256]
}
# Set the repeated k fold, number of epochs and model patience
random_state = 42
repeated_k_fold = RepeatedKFold(n_splits=5, n_repeats=6, random_state=random_state)
n_epochs = 1000
patience = 10

# Valence

#### Random Search

In [29]:
random_research_results_dfnn_loc = "./results/DFNN/random_search_dfnn.csv"
lr, wd, dp, batch_size = run_random_search(random_research_results_dfnn_loc, mean_data, "valence", repeated_k_fold, param_grid,
                                           number_of_visual_features, prediction_head, random_state)

Running random search


KeyboardInterrupt: 

#### Fit Model

In [6]:
k_folds_results_valence = train_model_cv(model_function = prediction_head,mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="valence",
                                         lr=lr,wd=wd,dp=dp,batch=batch_size,number_of_visual_features=number_of_visual_features,n_epochs=n_epochs,
                                        patience=patience, device=device, save=True, save_loc= model_save_loc, multiple_folder = True)


Training fold 0
Best model updated
Epoch [1/1000] Train Loss: 4.50257 | Test Loss: 3.78351 | MAE: 3.78351 | R2: -17.97945 | Patience: 0/10 | LR: 0.00014500
Best model updated
Epoch [2/1000] Train Loss: 3.49415 | Test Loss: 2.54375 | MAE: 2.54375 | R2: -8.12151 | Patience: 0/10 | LR: 0.00019000
Best model updated
Epoch [3/1000] Train Loss: 2.47634 | Test Loss: 1.69295 | MAE: 1.69295 | R2: -3.48453 | Patience: 0/10 | LR: 0.00023500
Best model updated
Epoch [4/1000] Train Loss: 1.74926 | Test Loss: 1.23221 | MAE: 1.23221 | R2: -1.54799 | Patience: 0/10 | LR: 0.00028000
Best model updated
Epoch [5/1000] Train Loss: 1.28723 | Test Loss: 0.97605 | MAE: 0.97605 | R2: -0.64641 | Patience: 0/10 | LR: 0.00032500
Best model updated
Epoch [6/1000] Train Loss: 1.01217 | Test Loss: 0.83872 | MAE: 0.83872 | R2: -0.21816 | Patience: 0/10 | LR: 0.00037000
Best model updated
Epoch [7/1000] Train Loss: 0.84411 | Test Loss: 0.76248 | MAE: 0.76248 | R2: -0.03082 | Patience: 0/10 | LR: 0.00041500
Best mode

In [7]:
performance_valence = performance_model_cv(model_function = prediction_head, mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="valence",
                                           device=device, n_features=number_of_visual_features, drop_out_rate=0.5, model_dir=model_save_loc, batch_size=256,
                                           scale=True, save=True, save_loc= os.path.join(result_save_loc, "dfnn_valence.csv"), multiple_folder = True)

Evaluating fold 0
Evaluating fold 1
Evaluating fold 2
Evaluating fold 3
Evaluating fold 4
Evaluating fold 5
Evaluating fold 6
Evaluating fold 7
Evaluating fold 8
Evaluating fold 9
Evaluating fold 10
Evaluating fold 11
Evaluating fold 12
Evaluating fold 13
Evaluating fold 14
Evaluating fold 15
Evaluating fold 16
Evaluating fold 17
Evaluating fold 18
Evaluating fold 19
Evaluating fold 20
Evaluating fold 21
Evaluating fold 22
Evaluating fold 23
Evaluating fold 24
Evaluating fold 25
Evaluating fold 26
Evaluating fold 27
Evaluating fold 28
Evaluating fold 29


# Tense

#### Random Search

In [8]:
random_research_results_dfnn_loc = "./results/DFNN/random_search_dfnn.csv"
lr, wd, dp, batch_size = run_random_search(random_research_results_dfnn_loc, mean_data, "tense", repeated_k_fold, param_grid,
                                           number_of_visual_features, prediction_head, random_state)

Running random search


#### Fit Model

In [9]:
k_folds_results_tense = train_model_cv(model_function = prediction_head,mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="tense",
                                         lr=lr,wd=wd,dp=dp,batch=batch_size,number_of_visual_features=number_of_visual_features,n_epochs=n_epochs,
                                        patience=patience, device=device, save=True, save_loc= model_save_loc, multiple_folder = True)


Training fold 0
Best model updated
Epoch [1/1000] Train Loss: 3.50803 | Test Loss: 2.90132 | MAE: 2.90132 | R2: -9.89669 | Patience: 0/10 | LR: 0.00014500
Best model updated
Epoch [2/1000] Train Loss: 2.51627 | Test Loss: 1.70032 | MAE: 1.70032 | R2: -3.35585 | Patience: 0/10 | LR: 0.00019000
Best model updated
Epoch [3/1000] Train Loss: 1.54611 | Test Loss: 0.95505 | MAE: 0.95505 | R2: -0.81382 | Patience: 0/10 | LR: 0.00023500
Best model updated
Epoch [4/1000] Train Loss: 0.91993 | Test Loss: 0.70696 | MAE: 0.70696 | R2: -0.06093 | Patience: 0/10 | LR: 0.00028000
Epoch [5/1000] Train Loss: 0.71096 | Test Loss: 0.71986 | MAE: 0.71986 | R2: 0.06116 | Patience: 1/10 | LR: 0.00032500
Epoch [6/1000] Train Loss: 0.69610 | Test Loss: 0.73869 | MAE: 0.73869 | R2: 0.07001 | Patience: 2/10 | LR: 0.00037000
Best model updated
Epoch [7/1000] Train Loss: 0.68308 | Test Loss: 0.69055 | MAE: 0.69055 | R2: 0.16064 | Patience: 0/10 | LR: 0.00041500
Best model updated
Epoch [8/1000] Train Loss: 0.637

In [10]:
performance_tense = performance_model_cv(model_function = prediction_head, mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="tense",
                                           device=device, n_features=number_of_visual_features, drop_out_rate=0.5, model_dir=model_save_loc, batch_size=256,
                                           scale=True, save=True, save_loc= os.path.join(result_save_loc, "dfnn_tense.csv"), multiple_folder = True)

Evaluating fold 0
Evaluating fold 1
Evaluating fold 2
Evaluating fold 3
Evaluating fold 4
Evaluating fold 5
Evaluating fold 6
Evaluating fold 7
Evaluating fold 8
Evaluating fold 9
Evaluating fold 10
Evaluating fold 11
Evaluating fold 12
Evaluating fold 13
Evaluating fold 14
Evaluating fold 15
Evaluating fold 16
Evaluating fold 17
Evaluating fold 18
Evaluating fold 19
Evaluating fold 20
Evaluating fold 21
Evaluating fold 22
Evaluating fold 23
Evaluating fold 24
Evaluating fold 25
Evaluating fold 26
Evaluating fold 27
Evaluating fold 28
Evaluating fold 29


# Energetic 

#### Random Search

In [11]:
random_research_results_dfnn_loc = "./results/DFNN/random_search_dfnn.csv"
lr, wd, dp, batch_size = run_random_search(random_research_results_dfnn_loc, mean_data, "energetic", repeated_k_fold, param_grid,
                                           number_of_visual_features, prediction_head, random_state)

Running random search


#### Fit Model

In [12]:
k_folds_results_energetic = train_model_cv(model_function = prediction_head,mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="energetic",
                                         lr=lr,wd=wd,dp=dp,batch=batch_size,number_of_visual_features=number_of_visual_features,n_epochs=n_epochs,
                                        patience=patience, device=device, save=True, save_loc= model_save_loc, multiple_folder = True)


Training fold 0
Best model updated
Epoch [1/1000] Train Loss: 3.16007 | Test Loss: 1.60476 | MAE: 1.60476 | R2: -3.11518 | Patience: 0/10 | LR: 0.00014500
Best model updated
Epoch [2/1000] Train Loss: 1.43675 | Test Loss: 0.85990 | MAE: 0.85990 | R2: -0.26931 | Patience: 0/10 | LR: 0.00019000
Best model updated
Epoch [3/1000] Train Loss: 0.86119 | Test Loss: 0.59015 | MAE: 0.59015 | R2: 0.10553 | Patience: 0/10 | LR: 0.00023500
Best model updated
Epoch [4/1000] Train Loss: 0.55171 | Test Loss: 0.38303 | MAE: 0.38303 | R2: 0.66320 | Patience: 0/10 | LR: 0.00028000
Best model updated
Epoch [5/1000] Train Loss: 0.39361 | Test Loss: 0.37085 | MAE: 0.37085 | R2: 0.70544 | Patience: 0/10 | LR: 0.00032500
Epoch [6/1000] Train Loss: 0.36311 | Test Loss: 0.41611 | MAE: 0.41611 | R2: 0.64752 | Patience: 1/10 | LR: 0.00037000
Epoch [7/1000] Train Loss: 0.34827 | Test Loss: 0.38436 | MAE: 0.38436 | R2: 0.69323 | Patience: 2/10 | LR: 0.00041500
Best model updated
Epoch [8/1000] Train Loss: 0.33827

In [13]:
performance_energetic = performance_model_cv(model_function = prediction_head, mean_data=mean_data,repeated_k_fold=repeated_k_fold,target="energetic",
                                           device=device, n_features=number_of_visual_features, drop_out_rate=0.5, model_dir=model_save_loc, batch_size=256,
                                           scale=True, save=True, save_loc= os.path.join(result_save_loc, "dfnn_energetic.csv"), multiple_folder = True)

Evaluating fold 0
Evaluating fold 1
Evaluating fold 2
Evaluating fold 3
Evaluating fold 4
Evaluating fold 5
Evaluating fold 6
Evaluating fold 7
Evaluating fold 8
Evaluating fold 9
Evaluating fold 10
Evaluating fold 11
Evaluating fold 12
Evaluating fold 13
Evaluating fold 14
Evaluating fold 15
Evaluating fold 16
Evaluating fold 17
Evaluating fold 18
Evaluating fold 19
Evaluating fold 20
Evaluating fold 21
Evaluating fold 22
Evaluating fold 23
Evaluating fold 24
Evaluating fold 25
Evaluating fold 26
Evaluating fold 27
Evaluating fold 28
Evaluating fold 29
